# Eval Harness Design

An eval is a **measurement instrument**, and most disagreements about model quality turn
out to be disagreements about the instrument rather than about the models. A harness that
parses answers slightly differently, or uses a slightly different prompt, can move a
reported score by more than the gap between two model generations.

This notebook is about building an instrument you can trust: what the pieces are, which
of them silently change your numbers, and how to tell an implementation artefact from a
real capability difference.

Companions: [LLM-as-a-Judge](llm-as-judge.ipynb) for model-graded scoring,
[The Statistics of Evals](eval-statistics.ipynb) for error bars, and
[Benchmark Contamination](benchmark-contamination.ipynb) for why a good instrument can
still give an inflated reading.

## 1. What & Why

A benchmark score is a pipeline, not a fact:

```
dataset → prompt template → sampling params → raw generation
        → answer extraction → normalisation → scoring → aggregation → one number
```

Every arrow is a decision, and several of them can shift the final number by several
points. When two papers report different scores for the same model on the same
benchmark, the cause is usually one of these arrows — not a different model.

**Reach for a custom harness when:**

- You need to measure a capability no public benchmark covers.
- You need scores that are *comparable across your own model versions* — which is a
  much weaker requirement than comparability with published numbers, and much more
  achievable.
- You need to debug *why* a model fails, not just how often. A harness that only emits
  an accuracy number is a poor research tool.

**Use an existing harness when** you need comparability with published results. Adopting
`lm-evaluation-harness` or `inspect` is the only realistic way to match someone else's
numbers, because you are adopting all their arrow-decisions along with the dataset.

**The honest framing:** an eval measures *the model plus your harness*. Reporting the
harness — prompts, parser, sampling, version — is as much a part of the result as the
score, and its absence is why so many published numbers cannot be reproduced.

## 2. Mental Model

**A scientific instrument, with the calibration step everyone skips.**

Nobody reports a physical measurement without knowing the instrument's error bars, its
systematic biases, and what it reads on a known control. Evals are routinely reported
with none of those.

The three things a usable instrument needs:

- **A known noise floor.** Run the same model twice under your sampling settings. The
  difference is your noise floor, and any model comparison smaller than it is not a
  result. See [The Statistics of Evals](eval-statistics.ipynb).
- **A known systematic bias.** The parser that drops answers in an unexpected format is
  not producing a random error — it penalises particular models systematically.
- **A control.** What does your harness score for a model that always answers "A"? For
  one that echoes the question? If a degenerate baseline scores 40% on a 4-way
  multiple-choice eval, the eval's usable range is 40–100%, not 0–100%.

The failure mode this framing prevents: treating a number as a property of the model
when it is a property of the *apparatus*. A harness bug does not look like a bug — it
looks like a finding.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Task spec** | The full definition: dataset, prompt template, sampling params, parser, scorer, aggregation. Everything needed to reproduce a number. |
| **Prompt template** | How an item becomes a prompt. Small changes here move scores more than most people expect. |
| **Few-shot / k-shot** | Examples prepended to the prompt. `k` is part of the score: 5-shot and 0-shot numbers are not comparable. |
| **Answer extraction** | Recovering a structured answer from free text. The single largest source of harness-induced variance. |
| **Normalisation** | Case-folding, whitespace, stripping articles/punctuation, numeric canonicalisation before comparison. |
| **Log-likelihood scoring** | Instead of generating, score each candidate answer's likelihood and take the argmax. No parsing, but needs logprob access, and needs length normalisation. |
| **pass@k** | For generative tasks with a checker: probability that at least one of `k` samples is correct. Has an unbiased estimator that is *not* "fraction of items where any sample passed". |
| **Aggregation** | Micro (pool all items) versus macro (mean of per-subtask scores). These can rank models differently. |
| **Degenerate baseline** | The score of a trivial strategy (always "A", most-common-answer, empty string). Your eval's real floor. |
| **Prompt sensitivity** | The spread of scores across semantically equivalent prompts. Report it, or you are reporting one sample from a distribution. |
| **Capability vs propensity** | Whether the model *can* do something versus whether it *does* by default. Different prompts measure different things. |

## 4. Setup

Standard library and NumPy. Every example below simulates model outputs so the
harness-side effects are isolated — no model call is needed, and the effects being
demonstrated are properties of the harness rather than of any particular model.

In [1]:
# %pip install numpy

import re
import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — the parser is a bigger variable than the model

Twelve realistic model responses to a maths question whose answer is `72`. Three
reasonable extraction strategies, three different scores.

In [2]:
responses = [
    "The answer is 72.",
    "72",
    "So we get 72 apples in total.",
    "Answer: 72",
    "**72**",
    "The total is $72.00",
    "First 8 * 9 = 72, so the answer is 72.",
    "I calculate 36 * 2 = 72",
    "The answer is 7 2",                    # tokeniser artefact
    "72 apples",
    "Let me think. 8 boxes, 9 each. 8*9=72. Final answer: 72",
    "The answer is seventy-two.",
]
GOLD = "72"

NUMBER = r"-?\d[\d,]*(?:\.\d+)?"      # note: no trailing bare "." -- "72." must parse as 72

def parse_first_number(text):
    m = re.search(NUMBER, text)
    return m.group(0).replace(",", "") if m else None

def parse_last_number(text):
    m = re.findall(NUMBER, text)
    return m[-1].replace(",", "") if m else None

def parse_strict_format(text):
    m = re.search(rf"(?:answer|total)\s*(?:is)?\s*:?\s*\**({NUMBER})", text, re.I)
    return m.group(1).replace(",", "") if m else None

parsers = {
    "first number in text": parse_first_number,
    "last number in text": parse_last_number,
    "strict 'Answer: N'": parse_strict_format,
}

print(f"{'response':52} " + " ".join(f"{n[:14]:>15}" for n in parsers))
for r in responses:
    row = []
    for fn in parsers.values():
        got = fn(r)
        row.append(f"{('OK ' if got == GOLD else 'x  ') + str(got):>15}")
    print(f"{r[:50]:52} " + " ".join(row))

print()
for name, fn in parsers.items():
    score = sum(1 for r in responses if fn(r) == GOLD) / len(responses)
    print(f"{name:24} accuracy = {score:.0%}")

print("\nSame model, same 12 responses. The reported accuracy spans a wide range purely")
print("on parser choice -- far more than the gap between most model versions.")
print("Note which responses each parser fails: 'first number' is destroyed by")
print("chain-of-thought (it grabs an intermediate value), while the strict format")
print("silently penalises any model that was not told to use it.")

response                                              first number i  last number in  strict 'Answer
The answer is 72.                                              OK 72           OK 72           OK 72
72                                                             OK 72           OK 72         x  None
So we get 72 apples in total.                                  OK 72           OK 72         x  None
Answer: 72                                                     OK 72           OK 72           OK 72
**72**                                                         OK 72           OK 72         x  None
The total is $72.00                                         x  72.00        x  72.00         x  None
First 8 * 9 = 72, so the answer is 72.                          x  8           OK 72           OK 72
I calculate 36 * 2 = 72                                        x  36           OK 72         x  None
The answer is 7 2                                               x  7            x  2       

The lesson is not "use the last number" — it is that **the parser is part of the
measurement**, so it must be chosen deliberately, reported, and ideally validated by
hand-checking a sample of the items it marks wrong. A parser tuned on one model's output
style systematically disadvantages models that format differently.

### Example 2 — your eval's real floor is not zero

Before trusting a score, find out what trivial strategies get.

In [3]:
# A 4-way multiple-choice set with the mild answer-position skew real datasets have.
n_items = 2000
gold = rng.choice(4, size=n_items, p=[0.30, 0.28, 0.22, 0.20])

def score(preds):
    return float(np.mean(preds == gold))

baselines = {
    "always 'A'": np.zeros(n_items, dtype=int),
    "always the most common label": np.full(n_items, int(np.bincount(gold).argmax())),
    "uniform random": rng.choice(4, size=n_items),
    "random, matched to label freq": rng.choice(4, size=n_items, p=[0.30, 0.28, 0.22, 0.20]),
}
for name, preds in baselines.items():
    print(f"{name:32} {score(preds):.1%}")

print(f"\nA 'real' model scoring 45% on this eval is barely above always answering 'A'.")
print("The usable dynamic range is roughly 30-100%, not 0-100%, so a 5-point gain from")
print("38% to 43% is a much smaller real improvement than it looks.")

# Chance-corrected agreement makes the same point in one number.
def cohens_kappa_vs_chance(acc, p_chance):
    return (acc - p_chance) / (1 - p_chance)

print("\nchance-corrected score (kappa-style), against the 'always A' floor of 30%:")
for acc in (0.30, 0.45, 0.60, 0.90):
    print(f"  raw {acc:.0%} -> corrected {cohens_kappa_vs_chance(acc, 0.30):.1%}")

always 'A'                       30.2%
always the most common label     30.2%
uniform random                   22.4%
random, matched to label freq    26.9%

A 'real' model scoring 45% on this eval is barely above always answering 'A'.
The usable dynamic range is roughly 30-100%, not 0-100%, so a 5-point gain from
38% to 43% is a much smaller real improvement than it looks.

chance-corrected score (kappa-style), against the 'always A' floor of 30%:
  raw 30% -> corrected 0.0%
  raw 45% -> corrected 21.4%
  raw 60% -> corrected 42.9%
  raw 90% -> corrected 85.7%


### Example 3 — prompt sensitivity is usually larger than the effect you are measuring

Simulate one model evaluated under ten semantically equivalent prompt templates, then
compare the spread against a genuine 2-point capability difference between two models.

In [4]:
def run_eval(true_ability, prompt_effect, n=1000, seed=0):
    '''Each item is answered correctly with prob = ability + this prompt's offset.'''
    r = np.random.default_rng(seed)
    p = np.clip(true_ability + prompt_effect, 0.01, 0.99)
    return float(r.random(n).mean() * 0 + (r.random(n) < p).mean())

# Ten equivalent phrasings; each has its own small, arbitrary offset.
prompt_offsets = rng.normal(0, 0.035, size=10)

model_a = [run_eval(0.62, off, seed=i) for i, off in enumerate(prompt_offsets)]
model_b = [run_eval(0.64, off, seed=100 + i) for i, off in enumerate(prompt_offsets)]

print("model A (true ability 62%) across 10 equivalent prompts:")
print("  " + "  ".join(f"{s:.1%}" for s in sorted(model_a)))
print(f"  spread: {max(model_a) - min(model_a):.1%}")
print("\nmodel B (true ability 64%) across the same 10 prompts:")
print("  " + "  ".join(f"{s:.1%}" for s in sorted(model_b)))

print(f"\ntrue capability gap        : 2.0%")
print(f"prompt-induced spread in A : {100*(max(model_a) - min(model_a)):.1f}%")
pairs = [(a, b) for a in model_a for b in model_b]
flipped = sum(1 for a, b in pairs if a > b)
print(f"\nIf each model is evaluated on ONE INDEPENDENTLY chosen prompt, model A (the")
print(f"weaker model) is reported as better in {flipped} of {len(pairs)} "
      f"({flipped/len(pairs):.0%}) of prompt pairings.")
print("\nSo a single-prompt evaluation of a 2-point difference is close to a coin flip.")
print("Report a mean and spread over several templates, or hold the template fixed and")
print("say so -- but do not present one prompt's number as the model's ability.")

model A (true ability 62%) across 10 equivalent prompts:
  59.5%  59.9%  60.1%  61.3%  62.7%  63.0%  63.1%  64.3%  65.7%  69.0%
  spread: 9.5%

model B (true ability 64%) across the same 10 prompts:
  60.5%  60.6%  62.2%  62.9%  63.1%  64.3%  64.4%  64.5%  67.3%  71.1%

true capability gap        : 2.0%
prompt-induced spread in A : 9.5%

If each model is evaluated on ONE INDEPENDENTLY chosen prompt, model A (the
weaker model) is reported as better in 35 of 100 (35%) of prompt pairings.

So a single-prompt evaluation of a 2-point difference is close to a coin flip.
Report a mean and spread over several templates, or hold the template fixed and
say so -- but do not present one prompt's number as the model's ability.


### Example 4 — `pass@k`, and the estimator almost everyone gets wrong

For generative tasks with an automatic checker, `pass@k` is the probability that at least
one of `k` samples passes. Estimating it by *drawing k samples and checking* is
high-variance and biased when you reuse the same samples for several `k`. The unbiased
estimator uses `n > k` samples per item.

In [5]:
from math import comb

def pass_at_k_unbiased(n, c, k):
    '''Unbiased pass@k for an item with c passing samples out of n drawn (n >= k).

    1 - C(n-c, k) / C(n, k)  =  1 - P(all k drawn samples are failures)
    '''
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

# Truth: per-item probability that one sample is correct.
n_problems = 400
p_correct = rng.beta(1.2, 3.0, size=n_problems)     # most problems hard, a few easy
N_SAMPLES = 20

counts = rng.binomial(N_SAMPLES, p_correct)          # c passing samples per item

print(f"{'k':>3} {'unbiased pass@k':>17} {'naive (first k samples)':>25} {'true':>9}")
for k in (1, 2, 5, 10):
    unbiased = np.mean([pass_at_k_unbiased(N_SAMPLES, int(c), k) for c in counts])
    # naive: just take the first k of the drawn samples and ask if any passed
    naive = np.mean([1.0 if rng.binomial(k, p) > 0 else 0.0 for p in p_correct])
    true = np.mean(1 - (1 - p_correct) ** k)
    print(f"{k:>3} {unbiased:17.4f} {naive:25.4f} {true:9.4f}")

print("\nBoth estimators are aimed at the same quantity, but the naive one is a single")
print("Bernoulli draw per item and is correspondingly noisy; the unbiased estimator")
print("pools information from all 20 samples and lands much closer to the truth.")
print("\nThe practical rule: sample n >> k once, then compute every pass@k you want")
print("from those same n samples with the formula above. Never re-sample per k.")

  k   unbiased pass@k   naive (first k samples)      true
  1            0.3044                    0.3025    0.2998
  2            0.4732                    0.5025    0.4689
  5            0.7026                    0.6975    0.7032
 10            0.8354                    0.8300    0.8348

Both estimators are aimed at the same quantity, but the naive one is a single
Bernoulli draw per item and is correspondingly noisy; the unbiased estimator
pools information from all 20 samples and lands much closer to the truth.

The practical rule: sample n >> k once, then compute every pass@k you want
from those same n samples with the formula above. Never re-sample per k.


### Example 5 — micro and macro aggregation can rank models differently

A benchmark of subtasks with very unequal sizes. Micro-averaging pools every item; macro
averages the per-subtask scores. They answer different questions, and they disagree.

In [6]:
subtasks = {
    #  name          n_items  model_X  model_Y
    "grammar":        (5000,   0.90,    0.86),
    "arithmetic":     (300,    0.40,    0.72),
    "commonsense":    (250,    0.55,    0.70),
    "translation":    (200,    0.60,    0.75),
}

def micro(col):
    total = sum(n for n, _, _ in subtasks.values())
    return sum(n * (x, y)[col] for n, x, y in subtasks.values()) / total

def macro(col):
    scores = [(x, y)[col] for _, x, y in subtasks.values()]
    return sum(scores) / len(scores)

print(f"{'subtask':26} {'n':>6} {'model X':>9} {'model Y':>9}")
for name, (n, x, y) in subtasks.items():
    print(f"{name:26} {n:6d} {x:9.0%} {y:9.0%}")

n_all = sum(n for n, _, _ in subtasks.values())
print(f"\n{'MICRO (pool items)':26} {n_all:6d} {micro(0):9.1%} {micro(1):9.1%}"
      f"   -> winner: {'X' if micro(0) > micro(1) else 'Y'}")
print(f"{'MACRO (mean of subtasks)':26} {'':6} {macro(0):9.1%} {macro(1):9.1%}"
      f"   -> winner: {'X' if macro(0) > macro(1) else 'Y'}")

print("\nModel Y is better at three of four subtasks and much better at the hard ones;")
print("model X wins on micro purely because the 5000-item grammar subtask dominates the")
print("pool. Neither number is wrong -- they answer different questions ('how often is")
print("it right on a random item' vs 'how broadly capable is it').")
print("\nState which you used. A benchmark with unequal subtasks and an unstated")
print("aggregation is not a reproducible measurement.")

subtask                         n   model X   model Y
grammar                      5000       90%       86%
arithmetic                    300       40%       72%
commonsense                   250       55%       70%
translation                   200       60%       75%

MICRO (pool items)           5750     84.8%     84.2%   -> winner: X
MACRO (mean of subtasks)              61.3%     75.8%   -> winner: Y

Model Y is better at three of four subtasks and much better at the hard ones;
model X wins on micro purely because the 5000-item grammar subtask dominates the
pool. Neither number is wrong -- they answer different questions ('how often is
it right on a random item' vs 'how broadly capable is it').

State which you used. A benchmark with unequal subtasks and an unstated
aggregation is not a reproducible measurement.


## 6. Gotchas & Pitfalls

- **Tuning the parser on one model's outputs.** It then systematically penalises models
  that format differently, and the effect looks exactly like a capability gap. Validate
  the parser by hand-reading a sample of what it marks wrong.
- **Not measuring the degenerate baseline.** Example 2: if "always A" scores 30%, your
  eval's range is 30–100%. Every reported delta should be read against that floor.
- **One prompt, reported as the model's ability.** Example 3. At minimum, report the
  template. Better, report mean and spread over several.
- **Comparing k-shot numbers across different k.** 5-shot and 0-shot scores are different
  measurements. So are different few-shot *example sets*.
- **Silent parse failures scored as wrong.** An unparseable response and a wrong answer
  are different events. Count them separately — a jump in parse failures is a harness
  regression, not a capability regression.
- **Log-likelihood scoring without length normalisation.** Longer candidate answers have
  lower total logprob by construction; without normalisation you are measuring length.
- **Recomputing pass@k by re-sampling per k.** Example 4. Sample once with `n >> k`.
- **Unstated aggregation.** Example 5.
- **Evaluating on the same items you iterated on.** Every look at a test set spends a
  little of it. Keep a held-out split you look at rarely, and expect your dev-set numbers
  to be optimistic.
- **Fixing the harness mid-experiment and comparing across the fix.** Re-run everything,
  or version the harness and never compare across versions.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Comparability with published numbers | An established harness — `lm-evaluation-harness`, `inspect`, HELM. Adopt their decisions wholesale |
| Tracking your own model versions | A custom harness, held fixed and versioned. Internal comparability is far easier than external |
| Open-ended quality with no checkable answer | [LLM-as-a-Judge](llm-as-judge.ipynb), with its biases measured |
| Deciding whether a difference is real | [The Statistics of Evals](eval-statistics.ipynb) |
| A suspiciously high score | [Benchmark Contamination](benchmark-contamination.ipynb) |
| Agent/tool-use trajectories | Trajectory-level scoring — the arrows in this notebook apply, but the unit of evaluation is an episode, not a string |

**The honest position.** Most eval effort is best spent not on the scoring function but
on the two ends: a dataset that measures what you claim, and error bars that tell you
whether a difference is real. The middle — parsing, normalisation, aggregation — is where
the bugs live, and the way to control it is to hold it fixed and version it rather than
to perfect it.

A harness you wrote and control, applied consistently across your own models, is worth
more for research decisions than a published number you cannot reproduce.

## 8. Resources

- [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) — the de-facto standard harness; read `docs/task_guide.md` for how a task spec is structured.
- [Inspect](https://inspect.aisi.org.uk/) — the UK AI Safety Institute's eval framework, designed around solvers and scorers as separable components.
- [Holistic Evaluation of Language Models (HELM)](https://arxiv.org/abs/2211.09110) — the multi-metric, multi-scenario argument, and a large study of prompt sensitivity.
- [Evaluating Large Language Models Trained on Code](https://arxiv.org/abs/2107.03374) — Appendix A derives the unbiased `pass@k` estimator used in Example 4.
- [Quantifying Language Models' Sensitivity to Spurious Features in Prompt Design](https://arxiv.org/abs/2310.11324) — how much formatting alone moves scores.
- [Lessons from the Trenches on Reproducible Evaluation of Language Models](https://arxiv.org/abs/2405.14782) — the EleutherAI team on why published numbers disagree.